In [9]:
# === CÓDIGO PARA GENERAR Y GUARDAR LOS DATOS ===

# Importaciones necesarias (asegúrate de tenerlas al inicio del archivo)
import random
import pandas as pd
import numpy as np
from faker import Faker
import datetime
from datetime import timedelta
import os

# Inicializar Faker
fake = Faker('es_ES')

In [10]:
# Función para generar ventas
def generate_sales(num_records):
    # Productos con precios y popularidad realistas para productos canarios
    products_info = {
        "Queso Majorero": {"price_range": (12.50, 18.90), "popularity": 0.17},
        "Aceite de Oliva": {"price_range": (9.95, 16.50), "popularity": 0.12},
        "Vino Tinto Canario": {"price_range": (14.95, 28.50), "popularity": 0.15},
        "Dulce de Membrillo": {"price_range": (6.50, 9.90), "popularity": 0.08},
        "Gofio": {"price_range": (4.95, 8.50), "popularity": 0.10},
        "Miel de Palma": {"price_range": (9.95, 15.50), "popularity": 0.09},
        "Licor Plátano de Canarias": {"price_range": (15.95, 25.50), "popularity": 0.08},
        "Mojo Picón": {"price_range": (5.95, 8.50), "popularity": 0.10},
        "Almogrote": {"price_range": (6.95, 10.50), "popularity": 0.06},
        "Papas Negras de Canarias": {"price_range": (8.95, 12.50), "popularity": 0.05}
    }
    
    # Preparar listas de productos y pesos para selección ponderada
    products = list(products_info.keys())
    weights = [info["popularity"] for info in products_info.values()]
    
    # Distribución de cantidad por producto
    quantity_distribution = {
        "Queso Majorero": [1, 1, 1, 2, 2, 3],
        "Aceite de Oliva": [1, 1, 2, 2, 3],
        "Vino Tinto Canario": [1, 1, 2, 2, 3, 6],  # Ocasionalmente compran cajas
        "Dulce de Membrillo": [1, 1, 2, 2],
        "Gofio": [1, 2, 2, 3, 3],
        "Miel de Palma": [1, 1, 2],
        "Licor Plátano de Canarias": [1, 1, 1, 2],
        "Mojo Picón": [1, 1, 2, 2, 3],
        "Almogrote": [1, 1, 2],
        "Papas Negras de Canarias": [1, 2, 2, 3, 4]
    }
    
    sales_data = []
    
    for _ in range(num_records):
        # Selecciona producto según popularidad
        product = random.choices(products, weights=weights, k=1)[0]
        
        # Selecciona cantidad según distribución típica para este producto
        quantity = random.choice(quantity_distribution[product])
        
        # Precio aleatorio dentro del rango específico del producto
        min_price, max_price = products_info[product]["price_range"]
        price = round(random.uniform(min_price, max_price), 2)
        
        # Aplicar descuento en algunas ventas
        if random.random() < 0.15:  # 15% de probabilidad de descuento
            discount = random.uniform(0.05, 0.20)  # Descuento entre 5% y 20%
            price = round(price * (1 - discount), 2)
        
        total_sale = round(quantity * price, 2)
        
        # Fecha con variación estacional
        # Ciertos productos tienen más demanda en temporadas específicas
        month_weights = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # Distribución uniforme por defecto
        
        # Ajustes estacionales para algunos productos
        if product == "Vino Tinto Canario":
            # Más ventas en navidad y verano
            month_weights = [1, 1, 1, 1, 1.2, 1.5, 1.5, 1.5, 1.2, 1, 1.5, 2]
        elif product == "Miel de Palma":
            # Más ventas en invierno
            month_weights = [1.5, 1.5, 1.2, 1, 1, 1, 1, 1, 1, 1.2, 1.5, 1.8]
        
        # Genera fecha considerando la estacionalidad - CORREGIDO
        month = random.choices(range(1, 13), weights=month_weights, k=1)[0]
        year = 2024  # Cambiado a 2024 para ser coherente con el resto del dataset
        
        # Determinar el último día del mes seleccionado
        if month in [4, 6, 9, 11]:
            max_day = 30
        elif month == 2:
            # 2024 es bisiesto, por lo que febrero tiene 29 días
            max_day = 29
        else:
            max_day = 31
        
        # Generar un día aleatorio válido para ese mes
        day = random.randint(1, max_day)
        
        # Crear la fecha correctamente
        sale_date = datetime.date(year, month, day)
        
        sales_data.append([sale_date, product, quantity, price, total_sale])
    
    return pd.DataFrame(sales_data, columns=["Fecha", "Producto", "Cantidad", "Precio Unitario", "Total Venta"])

In [11]:
# Función para generar clientes
def generate_clients(num_records):
    clients_data = []
    
    # Configurar Faker para español
    fake_es = Faker('es_ES')
    
    # Distribución de edad más realista para compradores de productos gourmet
    age_distribution = [
        (18, 24, 0.08),  # 8% entre 18-24 años
        (25, 34, 0.22),  # 22% entre 25-34 años
        (35, 44, 0.28),  # 28% entre 35-44 años
        (45, 54, 0.25),  # 25% entre 45-54 años
        (55, 65, 0.12),  # 12% entre 55-65 años
        (66, 80, 0.05)   # 5% mayores de 65 años
    ]
    
    # Ciudades canarias y principales ciudades españolas con ponderación
    locations = {
        "Las Palmas de Gran Canaria": 0.12,
        "Santa Cruz de Tenerife": 0.10,
        "La Laguna": 0.08,
        "Arrecife": 0.04,
        "Puerto del Rosario": 0.03,
        "Santa Cruz de La Palma": 0.03,
        "Madrid": 0.15,
        "Barcelona": 0.12,
        "Valencia": 0.08,
        "Sevilla": 0.06,
        "Bilbao": 0.05,
        "Zaragoza": 0.04,
        "Málaga": 0.05,
        "Alicante": 0.03,
        "Extranjero": 0.02
    }
    
    # Distribución de frecuencia de compra más realista
    purchase_freq_distribution = {
        "Una compra": 0.45,         # 45% han comprado una sola vez
        "Ocasional": 0.30,          # 30% compran ocasionalmente
        "Recurrente": 0.18,         # 18% son compradores habituales
        "VIP": 0.07                 # 7% son clientes VIP
    }
    
    for client_id in range(1, num_records + 1):
        # Género con distribución realista
        gender = random.choices(["M", "F"], weights=[0.48, 0.52], k=1)[0]
        
        # Nombre según género
        if gender == "M":
            name = fake_es.name_male()
        else:
            name = fake_es.name_female()
        
        # Edad según distribución ponderada
        age_range = random.choices(age_distribution, weights=[w for _, _, w in age_distribution], k=1)[0]
        age = random.randint(age_range[0], age_range[1])
        
        # Ubicación según distribución ponderada
        location_list = list(locations.keys())
        location_weights = list(locations.values())
        location = random.choices(location_list, weights=location_weights, k=1)[0]
        
        # Frecuencia de compra según distribución
        purchase_freq_list = list(purchase_freq_distribution.keys())
        purchase_freq_weights = list(purchase_freq_distribution.values())
        purchase_freq = random.choices(purchase_freq_list, weights=purchase_freq_weights, k=1)[0]
        
        # Fecha de registro (algunos son clientes antiguos, otros nuevos)
        registration_date = fake.date_between(start_date='-2y', end_date='today')
        
        # Fuente de adquisición
        acquisition_sources = {
            "Búsqueda orgánica": 0.25,
            "Redes sociales": 0.20,
            "Recomendación": 0.18,
            "Publicidad pagada": 0.15,
            "Email marketing": 0.12,
            "Evento gastronómico": 0.10
        }
        acquisition = random.choices(
            list(acquisition_sources.keys()), 
            weights=list(acquisition_sources.values()), 
            k=1
        )[0]
        
        clients_data.append([
            client_id, 
            name, 
            age, 
            gender, 
            location, 
            purchase_freq, 
            registration_date,
            acquisition
        ])
    
    return pd.DataFrame(clients_data, columns=[
        "ID Cliente", 
        "Nombre", 
        "Edad", 
        "Género", 
        "Ubicación", 
        "Frecuencia de compra",
        "Fecha de registro",
        "Fuente de adquisición"
    ])

In [12]:
# Función para generar tráfico web
def generate_traffic(num_records):
    traffic_data = []
    
    # Páginas con probabilidades ajustadas (la página de inicio suele tener más tráfico)
    landing_pages = {
        "Página de Inicio": 0.35,
        "Queso Majorero": 0.08, 
        "Aceite de Oliva": 0.08,
        "Vino Tinto Canario": 0.09,
        "Dulce de Membrillo": 0.05,
        "Gofio": 0.06,
        "Miel de Palma": 0.05,
        "Licor Plátano de Canarias": 0.07,
        "Mojo Picón": 0.08,
        "Almogrote": 0.04,
        "Papas Negras de Canarias": 0.05
    }
    
    # Lista de páginas ponderada para el random.choices
    pages = list(landing_pages.keys())
    weights = list(landing_pages.values())
    
    for _ in range(num_records):
        visit_date = fake.date_this_year()
        landing_page = random.choices(pages, weights=weights, k=1)[0]
        source = random.choice(["Orgánico", "Pagado", "Redes Sociales", "Email", "Directo", "Referidos"])
        device = random.choice(["Móvil", "Desktop", "Tablet"])
        
        # Ajustes más realistas según fuente y dispositivo
        if source == "Orgánico":
            visit_duration = random.uniform(1.5, 6)  # Duración media
            bounce_rate = random.uniform(40, 70)  # Tasa de rebote media-alta
        elif source == "Pagado":
            visit_duration = random.uniform(2, 7)  # Duración media-alta
            bounce_rate = random.uniform(35, 65)  # Tasa de rebote media
        elif source == "Redes Sociales":
            visit_duration = random.uniform(1, 4)  # Duración baja-media
            bounce_rate = random.uniform(60, 85)  # Tasa de rebote alta
        elif source == "Email":
            visit_duration = random.uniform(2.5, 8)  # Duración alta
            bounce_rate = random.uniform(30, 50)  # Tasa de rebote baja
        else:  # Directo o Referidos
            visit_duration = random.uniform(2, 6)  # Duración media
            bounce_rate = random.uniform(35, 60)  # Tasa de rebote media-baja
        
        # Ajustes según dispositivo
        if device == "Móvil":
            visit_duration *= 0.8  # Menor duración en móvil
            bounce_rate *= 1.2  # Mayor tasa de rebote en móvil (con límite de 100%)
            bounce_rate = min(bounce_rate, 95)
        elif device == "Tablet":
            visit_duration *= 0.9  # Duración intermedia en tablet
        
        # Ajustes según página de aterrizaje
        if landing_page == "Página de Inicio":
            visit_duration *= 1.1  # Mayor navegación desde la página principal
            bounce_rate *= 0.85  # Menor rebote desde la página principal
            
        traffic_data.append([
            visit_date, 
            landing_page, 
            round(visit_duration, 2),  # Duración redondeada a 2 decimales
            round(bounce_rate, 2), 
            source, 
            device
        ])
    
    return pd.DataFrame(traffic_data, columns=["Fecha", "Página de Aterrizaje", "Duración (min)", "Tasa de Rebote", "Fuente", "Dispositivo"])

In [13]:
# Función para generar campañas
def generate_campaigns(num_records):
    campaigns_data = []
    
    for campaign_id in range(1, num_records + 1):
        campaign_type = random.choice(["Anuncios en Google", "Email Marketing", "Redes Sociales"])
        
        start_date = fake.date_this_year()
        
        # Ajusta la duración según el tipo de campaña
        if campaign_type == "Email Marketing":
            # Email marketing: 1-7 días
            delta_days = fake.random_int(min=1, max=7)
        elif campaign_type == "Redes Sociales":
            # Redes sociales: 1-14 días
            delta_days = fake.random_int(min=1, max=14)
        elif campaign_type == "Anuncios en Google":
            # Google Ads: 7-30 días
            delta_days = fake.random_int(min=7, max=30)
        
        end_date = start_date + timedelta(days=delta_days)
        
        # Ajusta los valores según el tipo de campaña para mayor realismo
        if campaign_type == "Email Marketing":
            budget = random.uniform(200, 1000)  # Presupuesto menor para email marketing
            impressions = random.randint(5000, 30000)
            clicks = random.randint(200, 1500)
        elif campaign_type == "Redes Sociales":
            budget = random.uniform(500, 1800)  # Presupuesto medio para redes sociales
            impressions = random.randint(8000, 45000)
            clicks = random.randint(400, 2500)
        else:  # Anuncios en Google
            budget = random.uniform(800, 2000)  # Presupuesto mayor para Google Ads
            impressions = random.randint(10000, 50000)
            clicks = random.randint(500, 3000)
        
        # Calcula conversiones y ROI con tasas realistas según el tipo de campaña
        if campaign_type == "Email Marketing":
            conversion_rate = random.uniform(0.05, 0.15)  # 5-15% para email (más alto)
        elif campaign_type == "Redes Sociales":
            conversion_rate = random.uniform(0.02, 0.08)  # 2-8% para redes sociales
        else:  # Anuncios en Google
            conversion_rate = random.uniform(0.03, 0.10)  # 3-10% para Google
            
        conversions = int(clicks * conversion_rate)
        roi = round(random.uniform(1.5, 4.5), 2)
        
        campaigns_data.append([campaign_id, campaign_type, start_date, end_date, round(budget, 2), 
                              impressions, clicks, conversions, roi])
    
    return pd.DataFrame(campaigns_data, columns=["Campaña ID", "Tipo de Campaña", "Fecha Inicio", 
                                               "Fecha Fin", "Presupuesto", "Impressions", "Clics", 
                                               "Conversiones", "ROI"])

In [14]:
# Función para generar inventario
def generate_inventory(num_records):
    # Información detallada de productos por categoría
    products_info = {
        "Queso Majorero": {
            "category": "Lácteos y quesos",
            "supplier": "Quesería Fuerteventura",
            "cost": 9.75,
            "price": 15.95,
            "min_stock": 20,
            "max_stock": 80,
            "shelf_life": 60  # días
        },
        "Aceite de Oliva": {
            "category": "Aceites y condimentos",
            "supplier": "Aceites Canarios S.L.",
            "cost": 7.50,
            "price": 12.95,
            "min_stock": 30,
            "max_stock": 100,
            "shelf_life": 365
        },
        "Vino Tinto Canario": {
            "category": "Bebidas",
            "supplier": "Bodegas Tenerife",
            "cost": 11.25,
            "price": 19.95,
            "min_stock": 40,
            "max_stock": 150,
            "shelf_life": 730
        },
        "Dulce de Membrillo": {
            "category": "Dulces y mermeladas",
            "supplier": "Conservas Isleñas",
            "cost": 4.80,
            "price": 7.95,
            "min_stock": 15,
            "max_stock": 60,
            "shelf_life": 180
        },
        "Gofio": {
            "category": "Cereales y harinas",
            "supplier": "Molinos Gran Canaria",
            "cost": 3.50,
            "price": 6.50,
            "min_stock": 30,
            "max_stock": 90,
            "shelf_life": 180
        },
        "Miel de Palma": {
            "category": "Dulces y mermeladas",
            "supplier": "Apicultores La Gomera",
            "cost": 7.95,
            "price": 13.50,
            "min_stock": 20,
            "max_stock": 70,
            "shelf_life": 730
        },
        "Licor Plátano de Canarias": {
            "category": "Bebidas",
            "supplier": "Destilerías Canarias",
            "cost": 11.75,
            "price": 19.95,
            "min_stock": 15,
            "max_stock": 60,
            "shelf_life": 1095
        },
        "Mojo Picón": {
            "category": "Aceites y condimentos",
            "supplier": "Salsas Canarias",
            "cost": 3.95,
            "price": 6.95,
            "min_stock": 25,
            "max_stock": 80,
            "shelf_life": 120
        },
        "Almogrote": {
            "category": "Aceites y condimentos",
            "supplier": "Salsas Gomeras",
            "cost": 4.95,
            "price": 8.50,
            "min_stock": 15,
            "max_stock": 50,
            "shelf_life": 90
        },
        "Papas Negras de Canarias": {
            "category": "Productos frescos",
            "supplier": "Agricultores Tenerife",
            "cost": 6.75,
            "price": 10.95,
            "min_stock": 40,
            "max_stock": 120,
            "shelf_life": 30
        }
    }
    
    products = list(products_info.keys())
    inventory_data = []
    
    # Aseguramos que cada producto solo aparezca una vez
    if num_records > len(products):
        # Si se piden más registros que productos disponibles, usamos todos los productos
        selected_products = products
    else:
        # Seleccionamos aleatoriamente un subconjunto de productos
        selected_products = random.sample(products, num_records)
    
    for product_id, product in enumerate(selected_products, start=1):
        product_data = products_info[product]
        
        # Stock inicial acorde al rango específico del producto
        min_stock = product_data["min_stock"]
        max_stock = product_data["max_stock"]
        initial_stock = random.randint(min_stock, max_stock)
        
        # Stock actual más realista (basado en ventas simuladas)
        # Los productos más caros o con menos vida útil suelen tener rotación más lenta
        shelf_life = product_data["shelf_life"]
        price = product_data["price"]
        
        # Factor de rotación (productos más caros y con más vida útil tienen menor rotación)
        rotation_factor = 1 - (0.3 * price / 30) - (0.2 * shelf_life / 365)
        rotation_factor = max(0.3, min(0.9, rotation_factor))  # Limitamos entre 0.3 y 0.9
        
        # Stock actual calculado con simulación de rotación
        sold_percentage = random.uniform(0.2, 0.8) * rotation_factor
        current_stock = round(initial_stock * (1 - sold_percentage))
        
        # Margen de beneficio
        cost = product_data["cost"]
        margin = round(((price - cost) / cost) * 100, 2)
        
        # Estado del stock
        stock_status = "Normal"
        if current_stock <= min_stock * 0.5:
            stock_status = "Bajo"
        elif current_stock == 0:
            stock_status = "Agotado"
        elif current_stock >= max_stock * 0.9:
            stock_status = "Exceso"
        
        # Fecha de reposición
        if stock_status in ["Bajo", "Agotado"]:
            restock_date = fake.date_between(start_date="today", end_date="+10d")
        else:
            restock_date = None
            
        # SKU (Stock Keeping Unit)
        category_code = ''.join([word[0] for word in product_data["category"].split()])
        sku = f"{category_code}-{product_id:03d}-{product.replace(' ', '')[0:3].upper()}"
        
        inventory_data.append([
            product_id,
            product,
            sku,
            product_data["category"],
            product_data["supplier"],
            initial_stock,
            current_stock,
            product_data["min_stock"],
            product_data["max_stock"],
            round(cost, 2),
            round(price, 2),
            margin,
            stock_status,
            restock_date,
            product_data["shelf_life"]
        ])
    
    return pd.DataFrame(inventory_data, columns=[
        "ID Producto",
        "Nombre",
        "SKU",
        "Categoría",
        "Proveedor",
        "Stock Inicial",
        "Stock Actual",
        "Stock Mínimo",
        "Stock Máximo",
        "Costo Unitario",
        "Precio Venta",
        "Margen (%)",
        "Estado Stock",
        "Fecha Reposición",
        "Vida Útil (días)"
    ])

In [15]:
# Función para generar satisfacción del cliente
def generate_customer_satisfaction(num_records, client_df=None):
    satisfaction_data = []
    
    # Productos con tendencias de satisfacción basadas en calidad típica
    product_satisfaction = {
        "Queso Majorero": {"avg_rating": 4.7, "std_dev": 0.4, "review_freq": 0.18},
        "Aceite de Oliva": {"avg_rating": 4.3, "std_dev": 0.5, "review_freq": 0.15},
        "Vino Tinto Canario": {"avg_rating": 4.5, "std_dev": 0.6, "review_freq": 0.22},
        "Dulce de Membrillo": {"avg_rating": 4.1, "std_dev": 0.7, "review_freq": 0.12},
        "Gofio": {"avg_rating": 3.9, "std_dev": 0.8, "review_freq": 0.14},
        "Miel de Palma": {"avg_rating": 4.6, "std_dev": 0.5, "review_freq": 0.17},
        "Licor Plátano de Canarias": {"avg_rating": 4.4, "std_dev": 0.6, "review_freq": 0.16},
        "Mojo Picón": {"avg_rating": 4.2, "std_dev": 0.7, "review_freq": 0.19},
        "Almogrote": {"avg_rating": 4.5, "std_dev": 0.5, "review_freq": 0.15},
        "Papas Negras de Canarias": {"avg_rating": 4.0, "std_dev": 0.8, "review_freq": 0.11}
    }
    
    # Plantillas de comentarios según calificación
    comment_templates = {
        5: [
            "Excelente producto, {aspecto_positivo}. Volveré a comprar sin duda.",
            "Estoy encantado/a con este {producto}, {aspecto_positivo}.",
            "{producto} de primera calidad, {aspecto_positivo}.",
            "El mejor {producto} que he probado, {aspecto_positivo}.",
            "Superó mis expectativas, {aspecto_positivo}."
        ],
        4: [
            "Muy buen producto, {aspecto_positivo}, aunque {aspecto_negativo}.",
            "Bastante satisfecho/a con el {producto}, {aspecto_positivo}.",
            "Buena relación calidad-precio, {aspecto_positivo}.",
            "Recomendable {producto}, {aspecto_positivo}.",
            "{producto} de calidad, pero {aspecto_negativo}."
        ],
        3: [
            "Producto correcto, {aspecto_positivo}, pero {aspecto_negativo}.",
            "Calidad media. {aspecto_positivo}, sin embargo {aspecto_negativo}.",
            "Esperaba más del {producto}, {aspecto_negativo}.",
            "No está mal, pero {aspecto_negativo}.",
            "Aceptable, aunque {aspecto_negativo}."
        ],
        2: [
            "No recomendaría este {producto}, {aspecto_negativo}.",
            "Decepcionante, {aspecto_negativo}.",
            "Por debajo de mis expectativas, {aspecto_negativo}.",
            "No repetiré la compra, {aspecto_negativo}.",
            "Mala relación calidad-precio, {aspecto_negativo}."
        ],
        1: [
            "Muy insatisfecho/a con este {producto}, {aspecto_negativo}.",
            "No compren este {producto}, {aspecto_negativo}.",
            "Experiencia terrible, {aspecto_negativo}.",
            "Completamente decepcionado/a, {aspecto_negativo}.",
            "Pésima calidad, {aspecto_negativo}."
        ]
    }
    
    # Aspectos positivos y negativos específicos por producto
    product_aspects = {
        "Queso Majorero": {
            "positivos": ["sabor intenso", "textura cremosa", "auténtico sabor canario", "perfecto para tapas", "envío rápido"],
            "negativos": ["precio algo elevado", "tamaño más pequeño de lo esperado", "envío tardó un poco", "el empaquetado podría mejorar"]
        },
        "Aceite de Oliva": {
            "positivos": ["sabor suave", "excelente para ensaladas", "botella de calidad", "color dorado perfecto", "aroma increíble"],
            "negativos": ["el tapón gotea un poco", "precio elevado para la cantidad", "esperaba un sabor más intenso"]
        },
        "Vino Tinto Canario": {
            "positivos": ["bouquet excelente", "perfecto con carnes", "sabor afrutado", "buena crianza", "embotellado de calidad"],
            "negativos": ["corcho un poco seco", "precio elevado", "esperaba más cuerpo", "el envío tardó en llegar"]
        },
        "Dulce de Membrillo": {
            "positivos": ["textura perfecta", "dulzor equilibrado", "envase práctico", "combina genial con quesos", "sabor natural"],
            "negativos": ["un poco seco", "demasiado dulce para mi gusto", "tamaño pequeño", "se desmorona al cortar"]
        },
        "Gofio": {
            "positivos": ["versátil en la cocina", "sabor tradicional", "buena calidad del grano", "envase hermético", "alto valor nutritivo"],
            "negativos": ["algo difícil de preparar", "sabor muy peculiar", "el envase se daña fácilmente", "textura a veces grumosa"]
        },
        "Miel de Palma": {
            "positivos": ["sabor único", "consistencia perfecta", "envase de calidad", "color precioso", "versátil en repostería"],
            "negativos": ["demasiado espesa", "envase difícil de manipular", "precio elevado", "sabor muy intenso para algunos platos"]
        },
        "Licor Plátano de Canarias": {
            "positivos": ["sabor auténtico a plátano", "botella elegante", "perfecto para postres", "graduación alcohólica ideal", "retrogusto agradable"],
            "negativos": ["muy dulce", "precio elevado", "la botella es difícil de servir", "aroma demasiado intenso"]
        },
        "Mojo Picón": {
            "positivos": ["picante en su punto", "receta tradicional", "versátil en la cocina", "envase práctico", "sabor auténtico canario"],
            "negativos": ["demasiado picante", "consistencia algo líquida", "el envase gotea", "se seca rápido una vez abierto"]
        },
        "Almogrote": {
            "positivos": ["intenso sabor a queso", "textura perfecta", "ideal para untar", "envase de calidad", "cantidad generosa"],
            "negativos": ["demasiado salado", "textura algo granulosa", "envase difícil de abrir", "picante excesivo"]
        },
        "Papas Negras de Canarias": {
            "positivos": ["sabor auténtico", "piel fina", "excelentes para guisos", "buena conservación", "textura harinosa perfecta"],
            "negativos": ["algunas papas dañadas", "tamaño irregular", "envío tardó en llegar", "precio elevado para la cantidad"]
        }
    }
    
    # Fechas de review realistas (la mayoría dentro de una semana tras la compra)
    current_date = datetime.date.today()
    start_date = current_date - datetime.timedelta(days=365)
    
    # Usar clientes reales si se proporciona el dataframe
    if client_df is not None and not client_df.empty:
        client_ids = client_df["ID Cliente"].tolist()
    else:
        client_ids = list(range(1, num_records + 1))
    
    # Lista de productos
    products = list(product_satisfaction.keys())
    
    for _ in range(num_records):
        # Cliente aleatorio
        client_id = random.choice(client_ids)
        
        # Selección ponderada de producto según frecuencia de reviews
        product_weights = [product_satisfaction[p]["review_freq"] for p in products]
        product = random.choices(products, weights=product_weights, k=1)[0]
        
        # Generación de calificación según distribución normal truncada
        avg = product_satisfaction[product]["avg_rating"]
        std = product_satisfaction[product]["std_dev"]
        raw_rating = random.normalvariate(avg, std)
        rating = max(1, min(5, round(raw_rating)))  # Truncar entre 1-5
        
        # Generación de comentario personalizado
        template = random.choice(comment_templates[rating])
        aspecto_positivo = random.choice(product_aspects[product]["positivos"])
        
        if rating < 5:
            aspecto_negativo = random.choice(product_aspects[product]["negativos"])
            comment = template.format(producto=product, aspecto_positivo=aspecto_positivo, aspecto_negativo=aspecto_negativo)
        else:
            comment = template.format(producto=product, aspecto_positivo=aspecto_positivo)
        
        # Fecha de review
        days_since_registration = random.randint(1, 365)
        review_date = current_date - datetime.timedelta(days=days_since_registration)
        
        # Comprador verificado (90% lo son)
        verified_buyer = random.choices([True, False], weights=[0.9, 0.1], k=1)[0]
        
        # ¿Recibió respuesta del vendedor?
        seller_response = None
        if rating <= 3 and random.random() < 0.8:  # 80% de respuestas a reviews de 3 o menos estrellas
            response_templates = [
                "Gracias por su feedback. Lamentamos que no haya quedado totalmente satisfecho/a. Nos pondremos en contacto para resolver el problema.",
                "Sentimos mucho su experiencia negativa. Hemos tomado nota y mejoraremos. ¿Podría contactarnos para ofrecerle una solución?",
                "Agradecemos su sinceridad. Estamos trabajando para mejorar y nos gustaría compensarle por esta experiencia.",
                "Gracias por sus comentarios. Nos tomamos muy en serio todas las opiniones para mejorar constantemente."
            ]
            seller_response = random.choice(response_templates)
            
        # ¿Compartió en redes sociales?
        shared_social = rating >= 4 and random.random() < 0.3  # 30% de probabilidad para ratings altos
        
        satisfaction_data.append([
            client_id, 
            product, 
            rating, 
            comment, 
            review_date, 
            verified_buyer,
            seller_response,
            shared_social
        ])
    
    return pd.DataFrame(satisfaction_data, columns=[
        "ID Cliente", 
        "Producto", 
        "Calificación", 
        "Comentarios", 
        "Fecha Review", 
        "Comprador Verificado",
        "Respuesta Vendedor",
        "Compartido en Redes"
    ])

In [16]:
# Crear directorio de datos si no existe
data_dir = '../data'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Generación de los registros
# Primero generamos clientes, ya que se utilizarán en la función de satisfacción
clients_df = generate_clients(100)

# Generamos los demás datasets
inventory_df = generate_inventory(10)  # Generamos los 10 productos
sales_df = generate_sales(200)  # Usamos los precios reales definidos en inventory
traffic_df = generate_traffic(150)
campaigns_df = generate_campaigns(50)

# Generamos satisfacción pasando el dataframe de clientes para usar IDs reales
satisfaction_df = generate_customer_satisfaction(100, clients_df)

# Guardar los archivos CSV
sales_file = '../data/sales_data.csv'
clients_file = '../data/clients_data.csv'
traffic_file = '../data/traffic_data.csv'
campaigns_file = '../data/campaigns_data.csv'
inventory_file = '../data/inventory_data.csv'
satisfaction_file = '../data/satisfaction_data.csv'

# Guardar todos los DataFrames en formato CSV
sales_df.to_csv(sales_file, index=False)
clients_df.to_csv(clients_file, index=False)
traffic_df.to_csv(traffic_file, index=False)
campaigns_df.to_csv(campaigns_file, index=False)
inventory_df.to_csv(inventory_file, index=False)
satisfaction_df.to_csv(satisfaction_file, index=False)

print(f"Generación de datos completada. Archivos guardados en {data_dir}")
print(f"Se generaron:")
print(f"- {len(sales_df)} registros de ventas")
print(f"- {len(clients_df)} perfiles de clientes")
print(f"- {len(traffic_df)} registros de tráfico web")
print(f"- {len(campaigns_df)} campañas de marketing")
print(f"- {len(inventory_df)} productos en inventario")
print(f"- {len(satisfaction_df)} reseñas de satisfacción")

Generación de datos completada. Archivos guardados en ../data
Se generaron:
- 200 registros de ventas
- 100 perfiles de clientes
- 150 registros de tráfico web
- 50 campañas de marketing
- 10 productos en inventario
- 100 reseñas de satisfacción
